# Лабораторная работа №1
## Полносвязные сети, обратное распространение ошибки и геометрия признакового пространства

Цель: понять, как полносвязные сети формируют разделяющие поверхности и внутренние представления данных, проанализировать не только accuracy, но и геометрию пространства признаков, устойчивость модели и поведение современных архитектур.

In [ ]:
# === ПРОВЕРКА СРЕДЫ ВЫПОЛНЕНИЯ ===
import torch
if torch.cuda.is_available():
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден, будет использован CPU (для моделей этой работы это некритично)")

In [ ]:
# ============================================================
# ЛАБОРАТОРНАЯ РАБОТА №1
# Полносвязные сети, обратное распространение ошибки
# и геометрия признакового пространства
# ============================================================

# === ИМПОРТЫ (Можно пользоваться другими библиотеками для ML) ===
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
import random
import copy

import time
import pandas as pd
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})
CLASS_COLORS = ['#2a78d6', '#eb6834']
CLASS_NAMES = ['класс 0 (круг)', 'класс 1 (кольцо)']
MODEL_COLORS = {'A': '#2a78d6', 'B': '#eb6834', 'C': '#1baf7a'}
PROBA_CMAP = LinearSegmentedColormap.from_list('proba', ['#2a78d6', '#f0efec', '#eb6834'])
DIV_CMAP = LinearSegmentedColormap.from_list('div', ['#104281', '#3987e5', '#f0efec', '#e34948', '#9e2b2b'])
ERR_CMAP = LinearSegmentedColormap.from_list('err', ['#f0efec', '#86b6ef', '#2a78d6', '#104281'])

In [ ]:
# ============================================================
# ЧАСТЬ 1. Линейно-неразделимый синтетический датасет
# ============================================================
# ЗАДАНИЕ:
# 1. Сгенерируйте датасет "круг vs кольцо":
#    - класс 0: точки внутри круга радиуса 0.8
#    - класс 1: точки в кольце (внутренний радиус 1, внешний 2)
#    - не менее 1000 объектов, баланс 1:1
# 2. Визуализируйте распределение
# 3. Разделите на train/val/test (70/15/15)
# 4. Обучите три модели:
#    - Модель A: <=2 слоя, <=2 нейрона, ReLU
#    - Модель B: 2-3 слоя, >2 нейронов, ReLU
#    - Модель C: как A, но с GELU
# 5. Постройте границы решений и сравните метрики
# ============================================================

# === РЕШЕНИЕ ===

# --- 1.1 Генерация датасета ---
def make_circle_ring(n=1500, r_circle=0.8, r_ring=(1.0, 2.0), seed=SEED):
    rng = np.random.default_rng(seed)
    n0 = n // 2
    n1 = n - n0
    r0 = np.sqrt(rng.uniform(0.0, r_circle ** 2, n0))
    r1 = np.sqrt(rng.uniform(r_ring[0] ** 2, r_ring[1] ** 2, n1))
    r = np.concatenate([r0, r1])
    phi = rng.uniform(0.0, 2 * np.pi, n)
    X = np.stack([r * np.cos(phi), r * np.sin(phi)], axis=1).astype(np.float32)
    y = np.concatenate([np.zeros(n0), np.ones(n1)]).astype(np.int64)
    perm = rng.permutation(n)
    return X[perm], y[perm]


X, y = make_circle_ring(n=1500)
print(f"X: {X.shape}, объектов по классам: {np.bincount(y)}")

fig, ax = plt.subplots(figsize=(6, 6))
for c in (0, 1):
    m = y == c
    ax.scatter(X[m, 0], X[m, 1], s=8, alpha=0.7, color=CLASS_COLORS[c], label=CLASS_NAMES[c])
for r in (0.8, 1.0, 2.0):
    ax.add_patch(plt.Circle((0, 0), r, fill=False, ls='--', lw=1, color='0.45'))
ax.set_aspect('equal')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Круг vs кольцо, 1500 точек')
ax.legend(loc='upper right')
plt.show()

In [ ]:
# --- 1.2 Разбиение 70/15/15 со стратификацией ---
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15 / 0.85, stratify=y_tmp, random_state=SEED)

for name, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f"{name:5s}: {len(yy):4d} объектов, доля класса 1 = {yy.mean():.3f}")


def to_tensor(a, dtype=torch.float32):
    return torch.as_tensor(np.asarray(a), dtype=dtype, device=device)


Xtr_t, ytr_t = to_tensor(X_train), to_tensor(y_train)
Xva_t, yva_t = to_tensor(X_val), to_tensor(y_val)
Xte_t, yte_t = to_tensor(X_test), to_tensor(y_test)

In [ ]:
# --- 1.3 Модели и обучение ---
ACTIVATIONS = {'relu': nn.ReLU, 'gelu': nn.GELU, 'tanh': nn.Tanh, 'silu': nn.SiLU}


class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=(16, 16), act='relu', out_dim=1):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), ACTIVATIONS[act]()]
            d = h
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(d, out_dim)

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


def make_model(hidden, act, seed=SEED):
    set_seed(seed)
    return MLP(hidden=hidden, act=act).to(device)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


@torch.no_grad()
def predict_proba(model, X):
    model.eval()
    X = X if torch.is_tensor(X) else to_tensor(X)
    return torch.sigmoid(model(X)).cpu().numpy()


def accuracy(model, X, y):
    y = y.cpu().numpy() if torch.is_tensor(y) else np.asarray(y)
    return accuracy_score(y, (predict_proba(model, X) > 0.5).astype(int))


def train_classifier(model, X_tr, y_tr, X_va, y_va, epochs=500, lr=1e-2, batch_size=64, seed=SEED):
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    n = len(X_tr)
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val, best_state = float('inf'), None
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, generator=gen).to(device)
        epoch_loss = torch.zeros((), device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            opt.step()
            epoch_loss += loss.detach() * len(idx)
        model.eval()
        with torch.no_grad():
            logits = model(X_va)
            val_loss = loss_fn(logits, y_va).item()
            val_acc = ((logits > 0).float() == y_va).float().mean().item()
        hist['train_loss'].append(epoch_loss.item() / n)
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)
        if val_loss < best_val:
            best_val, best_state = val_loss, copy.deepcopy(model.state_dict())
    if device.type == 'cuda':
        torch.cuda.synchronize()
    hist['time'] = time.perf_counter() - t0
    hist['best_epoch'] = int(np.argmin(hist['val_loss'])) + 1
    model.load_state_dict(best_state)
    return model, hist


CONFIGS = {
    'A': dict(hidden=(2, 2), act='relu'),
    'B': dict(hidden=(16, 16), act='relu'),
    'C': dict(hidden=(2, 2), act='gelu'),
}
EPOCHS = 500

models, histories = {}, {}
for name, cfg in CONFIGS.items():
    model = make_model(**cfg, seed=SEED)
    model, hist = train_classifier(model, Xtr_t, ytr_t, Xva_t, yva_t, epochs=EPOCHS)
    models[name], histories[name] = model, hist
    print(f"Модель {name}: слои {cfg['hidden']}, {cfg['act']:4s} | параметров: {count_params(model):3d} | "
          f"val acc = {accuracy(model, Xva_t, yva_t):.3f} | время: {hist['time']:.1f} c")

In [ ]:
# --- 1.4 Кривые обучения ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
epochs_axis = np.arange(1, EPOCHS + 1)
for name, hist in histories.items():
    label = f"{name}: {CONFIGS[name]['hidden']}, {CONFIGS[name]['act'].upper()}"
    axes[0].plot(epochs_axis, hist['train_loss'], color=MODEL_COLORS[name], label=label)
    axes[1].plot(epochs_axis, hist['val_acc'], color=MODEL_COLORS[name], label=label)
axes[0].set_yscale('log')
axes[0].set_title('Train loss (BCE)')
axes[1].set_title('Val accuracy')
axes[1].set_ylim(0.4, 1.02)
for ax in axes:
    ax.set_xlabel('эпоха')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 1.5 Границы решений (сетка 200x200) ---
def grid_proba(model, lim=2.5, n=200):
    xs = np.linspace(-lim, lim, n)
    xx, yy = np.meshgrid(xs, xs)
    proba = predict_proba(model, np.c_[xx.ravel(), yy.ravel()].astype(np.float32)).reshape(n, n)
    return xx, yy, proba


def plot_decision_boundary(ax, model, X_pts, y_pts, lim=2.5, title='', extra_pts=None):
    xx, yy, proba = grid_proba(model, lim)
    cs = ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 11), cmap=PROBA_CMAP, alpha=0.6)
    ax.contour(xx, yy, proba, levels=[0.5], colors='k', linewidths=1.6)
    for c in (0, 1):
        m = y_pts == c
        ax.scatter(X_pts[m, 0], X_pts[m, 1], s=12, color=CLASS_COLORS[c],
                   edgecolors='white', linewidths=0.4, label=CLASS_NAMES[c])
    if extra_pts is not None:
        Xe, ye = extra_pts
        for c in (0, 1):
            m = ye == c
            ax.scatter(Xe[m, 0], Xe[m, 1], s=22, marker='x', color=CLASS_COLORS[c], linewidths=1.2)
    ax.set_aspect('equal')
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(False)
    ax.set_title(title)
    return cs


def boundary_closed(model, lim=2.5):
    _, _, p = grid_proba(model, lim)
    border = np.concatenate([p[0], p[-1], p[:, 0], p[:, -1]])
    return bool((border > 0.5).all() and (p < 0.5).any())


fig, axes = plt.subplots(1, 3, figsize=(16, 5.3))
for ax, (name, model) in zip(axes, models.items()):
    cfg = CONFIGS[name]
    cs = plot_decision_boundary(ax, model, X_test, y_test,
                                title=f"Модель {name}: {cfg['hidden']}, {cfg['act'].upper()}\n"
                                      f"test acc = {accuracy(model, Xte_t, yte_t):.3f}")
axes[0].legend(loc='upper left', fontsize=8)
fig.colorbar(cs, ax=axes, shrink=0.8, label='P(класс 1)')
fig.suptitle('Границы решений (черная линия - P = 0.5), точки - test', y=0.98)
plt.show()

In [ ]:
# --- 1.6 Итоговая таблица ---
rows = []
for name, model in models.items():
    cfg, hist = CONFIGS[name], histories[name]
    rows.append({
        'Модель': name,
        'Скрытые слои': 'x'.join(map(str, cfg['hidden'])),
        'Активация': cfg['act'].upper(),
        'Параметров': count_params(model),
        'Train acc': accuracy(model, Xtr_t, ytr_t),
        'Val acc': accuracy(model, Xva_t, yva_t),
        'Test acc': accuracy(model, Xte_t, yte_t),
        'Лучшая эпоха': hist['best_epoch'],
        'Время обучения, с': hist['time'],
    })
results_part1 = pd.DataFrame(rows).set_index('Модель')
display(results_part1.round(3))

conf = {}
for name, model in models.items():
    conf[name] = confusion_matrix(y_test, (predict_proba(model, Xte_t) > 0.5).astype(int))
    print(f"Модель {name}, матрица ошибок (строки - истина, столбцы - прогноз):\n{conf[name]}")

### Выводы по части 1

In [ ]:
acc1 = results_part1['Test acc']
times = results_part1['Время обучения, с']
gap = (results_part1[['Train acc', 'Val acc', 'Test acc']].max(axis=1)
       - results_part1[['Train acc', 'Val acc', 'Test acc']].min(axis=1)).max()
closed = {name: boundary_closed(m) for name, m in models.items()}

text = []
for name, cfg in CONFIGS.items():
    cm = conf[name]
    s = (f"- Модель {name} ({cfg['hidden'][0]}x{cfg['hidden'][1]}, {cfg['act'].upper()}, "
         f"{count_params(models[name])} параметров): test accuracy {acc1[name]:.3f}. ")
    if closed[name]:
        s += "Область класса 0 замкнута вокруг круга."
    else:
        s += "Область класса 0 не замкнута и уходит за край графика."
    if cm[0, 1] + cm[1, 0] > 0:
        s += (f" Ошибок на тесте {cm[0, 1] + cm[1, 0]}: точек кольца, отнесенных к кругу - {cm[1, 0]}, "
              f"точек круга, отнесенных к кольцу - {cm[0, 1]}.")
    text.append(s)

text.append("- У ReLU-сети граница кусочно-линейная, она собирается из отрезков прямых. "
            "Два нейрона в слое дают мало изломов, поэтому обвести круг такой сетью трудно. "
            f"У B нейронов 16, граница состоит из большего числа отрезков, test accuracy {acc1['B']:.3f}.")
d = (acc1['C'] - acc1['A']) * 100
text.append(f"- Замена ReLU на GELU при том же числе параметров изменила test accuracy на {d:+.1f} п.п. "
            "GELU гладкая, поэтому граница у модели C может изгибаться, а не состоит из отрезков.")
if gap < 0.03:
    text.append(f"- Accuracy на train, val и test у каждой модели отличается не больше чем на {gap * 100:.1f} п.п., "
                "переобучения нет. Разница между моделями связана с архитектурой.")
else:
    text.append(f"- Разрыв между train и test доходит до {gap * 100:.1f} п.п., у части моделей есть переобучение.")
text.append(f"- Время обучения за {EPOCHS} эпох: " + ', '.join(f"{n} - {t:.1f} с" for n, t in times.items()) + ".")
if times.max() / times.min() < 1.5:
    text[-1] += " У таких маленьких сетей время почти не зависит от числа параметров, оно определяется числом шагов."
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 2. Внутреннее признаковое пространство
# ============================================================
# ЗАДАНИЕ:
# Извлечь эмбеддинги из предпоследнего слоя каждой модели,
# применить t-SNE и визуализировать, как сеть преобразует
# исходное пространство.
# ============================================================

# === РЕШЕНИЕ ===

# --- 2.1 Эмбеддинги предпоследнего слоя ---
@torch.no_grad()
def get_embeddings(model, X):
    model.eval()
    return model.features(to_tensor(X)).cpu().numpy()


emb_test = {name: get_embeddings(m, X_test) for name, m in models.items()}
dead = {}
for name, e in emb_test.items():
    dead[name] = int((np.abs(e).max(axis=0) < 1e-6).sum())
    print(f"Модель {name}: эмбеддинги {e.shape}, нейронов, которые на тесте всегда 0: {dead[name]}")

emb_B_tsne = TSNE(n_components=2, perplexity=20, init='pca', random_state=SEED).fit_transform(emb_test['B'])
pca_B = PCA(n_components=2, random_state=SEED).fit(emb_test['B'])
emb_B_pca = pca_B.transform(emb_test['B'])
print("PCA для B, доля дисперсии PC1 и PC2:", pca_B.explained_variance_ratio_.round(3))

In [ ]:
# --- 2.2 Исходное пространство и пространство признаков ---
def scatter_classes(ax, Z, y_pts, title, xlabel, ylabel):
    for c in (0, 1):
        m = y_pts == c
        ax.scatter(Z[m, 0], Z[m, 1], s=14, alpha=0.75, color=CLASS_COLORS[c],
                   edgecolors='white', linewidths=0.3, label=CLASS_NAMES[c])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def draw_head_line(ax, model):
    w = model.head.weight.detach().cpu().numpy().ravel()
    b = model.head.bias.item()
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    if abs(w[1]) > 1e-8:
        xs = np.linspace(*xlim, 100)
        ax.plot(xs, -(w[0] * xs + b) / w[1], 'k--', lw=1.5, label='w*h + b = 0')
    else:
        ax.axvline(-b / w[0], color='k', ls='--', lw=1.5, label='w*h + b = 0')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


fig, axes = plt.subplots(2, 3, figsize=(16, 10))
scatter_classes(axes[0, 0], X_test, y_test, 'Исходное пространство (x1, x2)', 'x1', 'x2')
axes[0, 0].set_aspect('equal')
for ax, name in zip(axes[0, 1:], ['A', 'C']):
    scatter_classes(ax, emb_test[name], y_test,
                    f"Модель {name} ({CONFIGS[name]['act'].upper()}): скрытое пространство (h1, h2)", 'h1', 'h2')
    ax.margins(0.08)
    draw_head_line(ax, models[name])
scatter_classes(axes[1, 0], emb_B_pca, y_test,
                f"Модель B: PCA 16D -> 2D ({pca_B.explained_variance_ratio_.sum():.0%} дисперсии)", 'PC1', 'PC2')
scatter_classes(axes[1, 1], emb_B_tsne, y_test, 'Модель B: t-SNE 16D -> 2D (perplexity=20)', 't-SNE 1', 't-SNE 2')

r_test = np.linalg.norm(X_test, axis=1)
with torch.no_grad():
    logit_B = models['B'](Xte_t).cpu().numpy()
scatter_classes(axes[1, 2], np.c_[r_test, logit_B], y_test, 'Модель B: логит на выходе от радиуса', 'радиус', 'логит')
axes[1, 2].axhline(0, color='k', lw=1, ls='--')
axes[1, 2].axvspan(0.8, 1.0, color='0.5', alpha=0.15, label='зазор 0.8-1.0')
axes[0, 0].legend(loc='upper right', fontsize=8)
axes[0, 1].legend(fontsize=8)
axes[1, 2].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.3 Линейная разделимость до и после сети ---
def linear_probe(F_train, F_test):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=1e4, max_iter=10000))
    clf.fit(F_train, y_train)
    return accuracy_score(y_test, clf.predict(F_test))


add_r2 = lambda Z: np.c_[Z, (Z ** 2).sum(axis=1)]
probe = {
    'Исходные признаки (x1, x2)': linear_probe(X_train, X_test),
    'Исходные + признак r^2 = x1^2 + x2^2': linear_probe(add_r2(X_train), add_r2(X_test)),
}
for name, m in models.items():
    probe[f"Эмбеддинги модели {name}"] = linear_probe(get_embeddings(m, X_train), get_embeddings(m, X_test))
probe = pd.Series(probe, name='Test acc логистической регрессии')
display(probe.round(3).to_frame())

### Выводы по части 2

In [ ]:
raw_acc, r2_acc = probe.iloc[0], probe.iloc[1]
probe_emb = {name: probe[f"Эмбеддинги модели {name}"] for name in models}
text = [f"- На исходных координатах логистическая регрессия дает {raw_acc:.3f}, классы линейно не разделимы. "
        f"Если добавить один признак r^2 = x1^2 + x2^2, та же модель дает {r2_acc:.3f}."]

spearman = pd.Series(r_test).corr(pd.Series(logit_B), method='spearman')
s = (f"- Модель B: логистическая регрессия на ее эмбеддингах дает {probe_emb['B']:.3f}, "
     f"на PCA две компоненты объясняют {pca_B.explained_variance_ratio_.sum():.0%} дисперсии. "
     f"Логит на выходе связан с радиусом точки (корреляция Спирмена {spearman:.2f})")
neg, pos = r_test[logit_B < 0], r_test[logit_B > 0]
if len(neg) and len(pos) and neg.max() < pos.min():
    s += f", знак меняется между r = {neg.max():.2f} и r = {pos.min():.2f}"
s += "."
if spearman > 0.9:
    s += " То есть сеть фактически выучила признак, похожий на расстояние до центра."
if dead['B']:
    s += f" {dead['B']} из 16 нейронов последнего слоя на тесте всегда выдают 0."
text.append(s)

for name in ['A', 'C']:
    e = emb_test[name]
    s = f"- Модель {name}: логистическая регрессия на эмбеддингах дает {probe_emb[name]:.3f}."
    if CONFIGS[name]['act'] == 'relu':
        z0 = (np.abs(e[y_test == 0]).max(1) < 1e-6).mean()
        z1 = (np.abs(e[y_test == 1]).max(1) < 1e-6).mean()
        if z0 + z1 > 0:
            s += (f" В точку (0, 0) отображаются {z0:.0%} точек круга и {z1:.0%} точек кольца, "
                  "ReLU обнуляет отрицательные значения.")
    if e[:, 0].std() > 1e-6 and e[:, 1].std() > 1e-6:
        corr = np.corrcoef(e.T)[0, 1]
        if abs(corr) > 0.95:
            s += f" Два нейрона почти линейно зависимы (корреляция {corr:.2f}), точки лежат почти на одной прямой."
    text.append(s)

best_emb = max(probe_emb, key=probe_emb.get)
if probe_emb[best_emb] > raw_acc + 0.2:
    text.append(f"- Скрытые слои нелинейно деформируют плоскость, и после них классы разделяются прямой: "
                f"accuracy линейной модели растет с {raw_acc:.3f} до {probe_emb[best_emb]:.3f} "
                f"(лучше всего у модели {best_emb}). Последний слой сети просто проводит эту прямую.")
text.append("- t-SNE сохраняет только локальную структуру, поэтому расстояния между облаками на нем не сравниваются, "
            "вывод сделан по PCA и логистической регрессии.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 3. Шум, выбросы и устойчивость
# ============================================================
# ЗАДАНИЕ:
# 1. Добавить гауссов шум к данным
# 2. Измерить падение accuracy при разных уровнях шума
# 3. Построить график: уровень шума -> accuracy
# ============================================================

# === РЕШЕНИЕ ===

# --- 3.1 Гауссов шум на тесте (модели обучены на чистых данных) ---
SIGMAS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
N_REPEATS = 20
rng = np.random.default_rng(SEED)

noise_acc = {name: np.zeros((len(SIGMAS), N_REPEATS)) for name in list(models) + ['rule']}
for i, sigma in enumerate(SIGMAS):
    for rep in range(N_REPEATS):
        X_noisy = (X_test + rng.normal(0.0, sigma, size=X_test.shape)).astype(np.float32)
        for name, m in models.items():
            noise_acc[name][i, rep] = accuracy(m, X_noisy, y_test)
        noise_acc['rule'][i, rep] = accuracy_score(y_test, (np.linalg.norm(X_noisy, axis=1) > 0.9).astype(int))

fig, ax = plt.subplots(figsize=(8, 4.8))
for name in models:
    mean, std = noise_acc[name].mean(1), noise_acc[name].std(1)
    ax.plot(SIGMAS, mean, 'o-', color=MODEL_COLORS[name], ms=6,
            label=f"{name}: {CONFIGS[name]['hidden']}, {CONFIGS[name]['act'].upper()}")
    ax.fill_between(SIGMAS, mean - std, mean + std, color=MODEL_COLORS[name], alpha=0.15)
ax.plot(SIGMAS, noise_acc['rule'].mean(1), 's--', color='0.35', ms=5, lw=1.5, label='правило r > 0.9')
ax.axhline(0.5, color='0.6', lw=1, ls=':')
ax.set_xlabel('sigma шума')
ax.set_ylabel('test accuracy')
ax.set_title(f'Устойчивость к шуму (среднее и std по {N_REPEATS} повторам)')
ax.legend()
plt.show()

noise_df = pd.DataFrame({('правило r > 0.9' if k == 'rule' else f'Модель {k}'): v.mean(1)
                         for k, v in noise_acc.items()}, index=pd.Index(SIGMAS, name='sigma'))
display(noise_df.round(3))

In [ ]:
# --- 3.2 Выбросы: точки в кольце 2.5 <= r <= 4 со случайными метками в train и val ---
def make_outliers(n, rng, r_range=(2.5, 4.0)):
    r = np.sqrt(rng.uniform(r_range[0] ** 2, r_range[1] ** 2, n))
    phi = rng.uniform(0.0, 2 * np.pi, n)
    Xo = np.stack([r * np.cos(phi), r * np.sin(phi)], axis=1).astype(np.float32)
    yo = rng.integers(0, 2, n).astype(np.int64)
    return Xo, yo


OUTLIER_FRACS = [0.0, 0.05, 0.1, 0.2, 0.3]
outlier_acc = {name: [] for name in CONFIGS}
outlier_last = {}
rng_out = np.random.default_rng(SEED + 1)
for frac in OUTLIER_FRACS:
    Xo_tr, yo_tr = make_outliers(int(frac * len(X_train)), rng_out)
    Xo_va, yo_va = make_outliers(int(frac * len(X_val)), rng_out)
    Xtr_c, ytr_c = np.vstack([X_train, Xo_tr]), np.concatenate([y_train, yo_tr])
    Xva_c, yva_c = np.vstack([X_val, Xo_va]), np.concatenate([y_val, yo_va])
    for name, cfg in CONFIGS.items():
        if frac == 0.0:
            m = models[name]
        else:
            m = make_model(**cfg, seed=SEED)
            m, _ = train_classifier(m, to_tensor(Xtr_c), to_tensor(ytr_c),
                                    to_tensor(Xva_c), to_tensor(yva_c), epochs=EPOCHS)
        outlier_acc[name].append(accuracy(m, Xte_t, yte_t))
        if frac == OUTLIER_FRACS[-1]:
            outlier_last[name] = m
    print(f"выбросов {frac:.0%}: " + ', '.join(f"{k}={v[-1]:.3f}" for k, v in outlier_acc.items()))

fig = plt.figure(figsize=(17, 5.2))
ax0 = fig.add_subplot(1, 3, 1)
for name in CONFIGS:
    ax0.plot(np.array(OUTLIER_FRACS) * 100, outlier_acc[name], 'o-', color=MODEL_COLORS[name], ms=6,
             label=f"{name}: {CONFIGS[name]['hidden']}, {CONFIGS[name]['act'].upper()}")
ax0.set_xlabel('доля выбросов в train/val, %')
ax0.set_ylabel('test accuracy')
ax0.set_ylim(0.4, 1.02)
ax0.set_title('Влияние выбросов')
ax0.legend()
for k, name in enumerate(['A', 'B']):
    ax = fig.add_subplot(1, 3, k + 2)
    plot_decision_boundary(ax, outlier_last[name], X_test, y_test, lim=4.3, extra_pts=(Xo_tr, yo_tr),
                           title=f"Модель {name}, {OUTLIER_FRACS[-1]:.0%} выбросов (x)")
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.3 Уменьшение числа нейронов: 2 скрытых слоя ширины w, 3 seed ---
WIDTHS = [1, 2, 3, 4, 8, 16]
WIDTH_SEEDS = [0, 1, 2]
width_acc = {act: np.zeros((len(WIDTHS), len(WIDTH_SEEDS))) for act in ('relu', 'gelu')}
for act in width_acc:
    for i, w in enumerate(WIDTHS):
        for j, s in enumerate(WIDTH_SEEDS):
            m = make_model(hidden=(w, w), act=act, seed=s)
            m, _ = train_classifier(m, Xtr_t, ytr_t, Xva_t, yva_t, epochs=EPOCHS, seed=s)
            width_acc[act][i, j] = accuracy(m, Xte_t, yte_t)
        print(f"{act.upper():4s} w={w:2d}: test acc по seed = {np.round(width_acc[act][i], 3)}")

fig, ax = plt.subplots(figsize=(8, 4.8))
for act, color in [('relu', MODEL_COLORS['B']), ('gelu', MODEL_COLORS['C'])]:
    acc = width_acc[act]
    ax.plot(WIDTHS, acc.mean(1), 'o-', color=color, ms=6, label=f'{act.upper()}, среднее по 3 seed')
    for j in range(len(WIDTH_SEEDS)):
        ax.scatter(WIDTHS, acc[:, j], s=16, color=color, alpha=0.4)
ax.set_xscale('log', base=2)
ax.set_xticks(WIDTHS)
ax.set_xticklabels(WIDTHS)
ax.set_ylim(0.4, 1.02)
ax.set_xlabel('нейронов в каждом из 2 скрытых слоев')
ax.set_ylabel('test accuracy')
ax.set_title('Влияние числа нейронов (точки - отдельные seed)')
ax.legend()
plt.show()

In [ ]:
# --- 3.4 Что влияет сильнее (падение accuracy модели B) ---
base = noise_acc['B'][0].mean()
w16, w2, w1 = WIDTHS.index(16), WIDTHS.index(2), WIDTHS.index(1)
factors = pd.DataFrame([
    ('Шум входа', 'sigma = 0.3', base - noise_acc['B'][SIGMAS.index(0.3)].mean()),
    ('Шум входа', 'sigma = 1.0', base - noise_acc['B'][SIGMAS.index(1.0)].mean()),
    ('Выбросы', '30% точек', outlier_acc['B'][0] - outlier_acc['B'][-1]),
    ('Меньше нейронов', '16 -> 2, ReLU', width_acc['relu'][w16].mean() - width_acc['relu'][w2].mean()),
    ('Меньше нейронов', '16 -> 1, ReLU', width_acc['relu'][w16].mean() - width_acc['relu'][w1].mean()),
], columns=['Фактор', 'Уровень', 'Падение accuracy, п.п.'])
factors['Падение accuracy, п.п.'] *= 100
display(factors.round(1))

fig, ax = plt.subplots(figsize=(8, 3.6))
labels = [f'{f}: {l}' for f, l in zip(factors['Фактор'], factors['Уровень'])]
ax.barh(labels[::-1], factors['Падение accuracy, п.п.'][::-1], color='#2a78d6', height=0.55)
ax.set_xlabel('падение test accuracy, п.п.')
ax.set_title('Сравнение факторов (модель B)')
plt.show()

### Выводы по части 3

In [ ]:
s03, s10 = SIGMAS.index(0.3), SIGMAS.index(1.0)
mean_noisy = {name: noise_acc[name][1:].mean() for name in models}
robust_name = max(mean_noisy, key=mean_noisy.get)
diff_rule = np.abs(noise_acc['B'].mean(1) - noise_acc['rule'].mean(1)).max()

text = [f"- Шум: с ростом sigma accuracy падает у всех моделей. При sigma = 0.3: "
        + ', '.join(f"{n} - {noise_acc[n][s03].mean():.3f}" for n in models)
        + f"; при sigma = 1.0: " + ', '.join(f"{n} - {noise_acc[n][s10].mean():.3f}" for n in models)
        + f". В среднем по sigma > 0 устойчивее всех модель {robust_name}."]
if diff_rule < 0.03:
    text.append(f"- Кривая модели B почти совпадает с правилом r > 0.9 (разница не больше {diff_rule * 100:.1f} п.п.). "
                "Значит, падение связано с самими данными: зазор между классами всего 0.2, и шум переносит точки через "
                "границу. Модель, обученная на чистых данных, вряд ли сработает лучше этого правила.")
else:
    text.append(f"- Модель B отличается от правила r > 0.9 до {diff_rule * 100:.1f} п.п., то есть часть потерь "
                "связана с формой ее границы, а не только с данными.")

drops = {n: (outlier_acc[n][0] - outlier_acc[n][-1]) * 100 for n in CONFIGS}
s = "- Выбросы (30% точек в train и val), изменение accuracy: " + ', '.join(f"{n} {-v:+.1f} п.п." for n, v in drops.items())
if drops['B'] < 1:
    s += (" Модель B выбросы почти не задели: ей хватает нейронов, чтобы подогнать их отдельными областями "
          "на краях, не трогая границу между кругом и кольцом.")
small = [n for n in ('A', 'C') if drops[n] > 3]
if small:
    s += f" Маленьким моделям ({', '.join(small)}) выбросы мешают сильнее, они сдвигают их границу."
text.append(s)


def min_stable_width(act):
    ok = [w for w, row in zip(WIDTHS, width_acc[act]) if row.min() >= 0.99]
    return ok[0] if ok else None


for act in ('relu', 'gelu'):
    w_ok = min_stable_width(act)
    row2 = width_acc[act][w2]
    s = (f"- Число нейронов, {act.upper()}: при ширине 2 accuracy по seed от {row2.min():.3f} до {row2.max():.3f}, "
         f"при ширине 1 в среднем {width_acc[act][w1].mean():.3f}. ")
    s += (f"Начиная с ширины {w_ok} все seed дают не меньше 0.99." if w_ok
          else "Ни при какой ширине не все seed дают 0.99.")
    text.append(s)
all_a = np.r_[acc1['A'], width_acc['relu'][w2]]
text.append(f"- Модель A из части 1 (seed {SEED}) получила {acc1['A']:.3f}, а та же архитектура на seed 0-2 - "
            f"{width_acc['relu'][w2].min():.3f}-{width_acc['relu'][w2].max():.3f}. "
            + ("Для маленьких сетей результат сильно зависит от инициализации." if np.ptp(all_a) > 0.05
               else "Результат от seed почти не зависит."))

top = factors.sort_values('Падение accuracy, п.п.', ascending=False).iloc[0]
FACTOR_PHRASE = {'Шум входа': 'шум входа', 'Выбросы': 'доля выбросов', 'Меньше нейронов': 'уменьшение числа нейронов'}
top_phrase = f"{FACTOR_PHRASE[top['Фактор']]} ({top['Уровень']})"
text.append(f"- Сильнее всего на модель B влияет {top_phrase}: accuracy падает на {top['Падение accuracy, п.п.']:.1f} п.п. "
            "Шум ухудшает сами данные и мешает всем моделям, "
            "нехватка нейронов ломает модель уже на чистых данных, а выбросы опасны в основном для маленьких сетей.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 4. Kolmogorov-Arnold Networks (KAN)
# ============================================================
# ЗАДАНИЕ:
# Сравнить классический MLP с KAN (Kolmogorov-Arnold Network)
# на аппроксимации сложной многомерной функции.
# KAN заменяет веса на обучаемые 1D-функции активации на ребрах.
# ============================================================

# === РЕШЕНИЕ ===

# --- 4.1 Данные: f(x, y) = sin(pi*x)*cos(pi*y) + x^2 - y^2 на [-1, 1]^2 ---
def target_fn(x, y):
    return np.sin(np.pi * x) * np.cos(np.pi * y) + x ** 2 - y ** 2


def make_regression(n, seed):
    rng = np.random.default_rng(seed)
    P = rng.uniform(-1, 1, size=(n, 2)).astype(np.float32)
    return P, target_fn(P[:, 0], P[:, 1]).astype(np.float32)


P_train, f_train = make_regression(2000, SEED)
P_val, f_val = make_regression(500, SEED + 1)
P_test, f_test = make_regression(2000, SEED + 2)
Ptr_t, ftr_t = to_tensor(P_train), to_tensor(f_train)
Pva_t, fva_t = to_tensor(P_val), to_tensor(f_val)
Pte_t, fte_t = to_tensor(P_test), to_tensor(f_test)
print(f"train {P_train.shape}, val {P_val.shape}, test {P_test.shape}, "
      f"диапазон f: [{f_train.min():.2f}, {f_train.max():.2f}]")

g = np.linspace(-1, 1, 100)
GX, GY = np.meshgrid(g, g)
GF = target_fn(GX, GY)
F_LIM = np.abs(GF).max()
F_LEVELS = np.linspace(-F_LIM, F_LIM, 31)
grid_t = to_tensor(np.c_[GX.ravel(), GY.ravel()])

fig = plt.figure(figsize=(12, 4.8))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.plot_surface(GX, GY, GF, cmap=DIV_CMAP, vmin=-F_LIM, vmax=F_LIM, linewidth=0)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('f(x, y) = sin(pi x) cos(pi y) + x^2 - y^2')
ax = fig.add_subplot(1, 2, 2)
im = ax.contourf(GX, GY, GF, levels=F_LEVELS, cmap=DIV_CMAP)
fig.colorbar(im, ax=ax, ticks=np.arange(-1.5, 1.6, 0.5))
ax.set_aspect('equal')
ax.grid(False)
ax.set_title('Линии уровня f')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2 Упрощенный KAN на B-сплайнах ---
class KANLinear(nn.Module):
    def __init__(self, in_features, out_features, grid_size=8, spline_order=3, grid_range=(-1.0, 1.0)):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.G, self.k = grid_size, spline_order
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32) * h + grid_range[0]
        self.register_buffer('grid', grid.expand(in_features, -1).contiguous())
        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.base_weight, a=5 ** 0.5)
        self.spline_weight = nn.Parameter(0.1 * torch.randn(out_features, in_features, grid_size + spline_order))

    def b_splines(self, x):
        g = self.grid
        x = x.unsqueeze(-1)
        B = ((x >= g[:, :-1]) & (x < g[:, 1:])).to(x.dtype)
        for p in range(1, self.k + 1):
            B = ((x - g[:, :-(p + 1)]) / (g[:, p:-1] - g[:, :-(p + 1)]) * B[..., :-1]
                 + (g[:, p + 1:] - x) / (g[:, p + 1:] - g[:, 1:-p]) * B[..., 1:])
        return B

    def forward(self, x):
        base = F.silu(x) @ self.base_weight.T
        spline = torch.einsum('nik,oik->no', self.b_splines(x), self.spline_weight)
        return base + spline

    @torch.no_grad()
    def edge_functions(self, x1d):
        X = x1d.unsqueeze(1).expand(-1, self.in_features)
        spline = torch.einsum('nik,oik->nio', self.b_splines(X), self.spline_weight)
        return F.silu(X).unsqueeze(-1) * self.base_weight.T.unsqueeze(0) + spline

    @torch.no_grad()
    def update_grid(self, x, margin=0.01):
        old_spline = torch.einsum('nik,oik->nio', self.b_splines(x), self.spline_weight)
        lo, hi = x.min(0).values - margin, x.max(0).values + margin
        steps = torch.arange(-self.k, self.G + self.k + 1, dtype=x.dtype, device=x.device)
        self.grid.copy_(lo[:, None] + steps[None, :] * ((hi - lo) / self.G)[:, None])
        A = self.b_splines(x).permute(1, 0, 2)
        coef = torch.linalg.lstsq(A, old_spline.permute(1, 0, 2)).solution
        self.spline_weight.copy_(coef.permute(2, 0, 1))


class KAN(nn.Module):
    def __init__(self, widths=(2, 5, 1), grid_size=8, spline_order=3):
        super().__init__()
        self.widths = widths
        self.layers = nn.ModuleList(KANLinear(a, b, grid_size, spline_order)
                                    for a, b in zip(widths[:-1], widths[1:]))

    def forward(self, x, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x.squeeze(-1)


_B = KANLinear(1, 1).b_splines(torch.linspace(-0.999, 0.999, 7).unsqueeze(1))
print("сумма B_k(x) на [-1, 1]:", _B.sum(-1).ravel().numpy().round(6))

In [ ]:
# --- 4.3 Обучение: MLP 3x16, маленький MLP с числом параметров как у KAN, KAN ---
class RegMLP(MLP):
    def forward(self, x, update_grid=False):
        return super().forward(x)


def train_regressor(model, epochs=2000, lr=1e-2, grid_update_epochs=()):
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {'train_mse': [], 'val_mse': []}
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        if epoch in grid_update_epochs:
            model(Ptr_t, update_grid=True)
        opt.zero_grad()
        loss = F.mse_loss(model(Ptr_t), ftr_t)
        loss.backward()
        opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            hist['val_mse'].append(F.mse_loss(model(Pva_t), fva_t).item())
        hist['train_mse'].append(loss.item())
    if device.type == 'cuda':
        torch.cuda.synchronize()
    hist['time'] = time.perf_counter() - t0
    return model, hist


REG_EPOCHS = 2000
reg_models, reg_hist = {}, {}
set_seed(SEED)
reg_models['MLP 3x16'] = RegMLP(hidden=(16, 16, 16), act='silu').to(device)
set_seed(SEED)
reg_models['MLP 10x10'] = RegMLP(hidden=(10, 10), act='silu').to(device)
set_seed(SEED)
reg_models['KAN [2, 5, 1]'] = KAN(widths=(2, 5, 1), grid_size=8, spline_order=3).to(device)
REG_COLORS = dict(zip(reg_models, ['#2a78d6', '#1baf7a', '#eb6834']))

for name, model in reg_models.items():
    upd = (100, 300, 600) if isinstance(model, KAN) else ()
    reg_models[name], reg_hist[name] = train_regressor(model, epochs=REG_EPOCHS, grid_update_epochs=upd)
    print(f"{name:14s} параметров: {count_params(model):4d} | время: {reg_hist[name]['time']:.1f} c")

In [ ]:
# --- 4.4 Сравнение: параметры, сходимость, качество на тесте ---
def first_epoch_below(curve, thr):
    idx = np.nonzero(np.asarray(curve) < thr)[0]
    return int(idx[0]) + 1 if len(idx) else None


fig, ax = plt.subplots(figsize=(9, 4.8))
for name, hist in reg_hist.items():
    ax.plot(np.arange(1, REG_EPOCHS + 1), hist['val_mse'], color=REG_COLORS[name],
            label=f"{name} ({count_params(reg_models[name])} пар.)")
    ax.plot(np.arange(1, REG_EPOCHS + 1), hist['train_mse'], color=REG_COLORS[name], lw=1, alpha=0.35)
ax.set_yscale('log')
ax.set_xlabel('эпоха')
ax.set_ylabel('MSE')
ax.set_title('Val MSE (толстые линии) и train MSE (тонкие)')
ax.legend()
plt.show()

reg_rows = []
for name, model in reg_models.items():
    model.eval()
    with torch.no_grad():
        pred = model(Pte_t)
        test_mse = F.mse_loss(pred, fte_t).item()
        test_max = (pred - fte_t).abs().max().item()
    h = reg_hist[name]
    reg_rows.append({
        'Модель': name,
        'Параметров': count_params(model),
        'Test MSE': test_mse,
        'Test RMSE': test_mse ** 0.5,
        'Макс. ошибка': test_max,
        'Эпох до val MSE < 1e-2': first_epoch_below(h['val_mse'], 1e-2),
        'Эпох до val MSE < 1e-3': first_epoch_below(h['val_mse'], 1e-3),
        'Эпох до val MSE < 1e-4': first_epoch_below(h['val_mse'], 1e-4),
        'Время на эпоху, мс': 1000 * h['time'] / REG_EPOCHS,
    })
results_part4 = pd.DataFrame(reg_rows).set_index('Модель')
with pd.option_context('display.float_format', '{:.3g}'.format):
    display(results_part4)

In [ ]:
# --- 4.5 Предсказания и ошибки на сетке ---
preds = {}
for name, model in reg_models.items():
    with torch.no_grad():
        preds[name] = model(grid_t).cpu().numpy().reshape(GX.shape)

err_max = max(np.abs(p - GF).max() for p in preds.values())
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
cs_f = axes[0, 0].contourf(GX, GY, GF, levels=F_LEVELS, cmap=DIV_CMAP)
axes[0, 0].set_title('Истинная f(x, y)')
for k, (name, P) in enumerate(preds.items(), start=1):
    axes[0, k].contourf(GX, GY, np.clip(P, -F_LIM, F_LIM), levels=F_LEVELS, cmap=DIV_CMAP)
    axes[0, k].set_title(f'Предсказание: {name}')
    im = axes[1, k].imshow(np.abs(P - GF), origin='lower', extent=(-1, 1, -1, 1),
                           cmap=ERR_CMAP, vmin=0, vmax=err_max)
    axes[1, k].set_title(f'|ошибка|: {name}\nmax = {np.abs(P - GF).max():.3f}')
fig.colorbar(cs_f, ax=axes[0, :], shrink=0.85, label='f', ticks=np.arange(-1.5, 1.6, 0.5))
fig.colorbar(im, ax=axes[1, 1:], shrink=0.85, label='|ошибка|')
ys = 0.5
xs_cut = np.linspace(-1, 1, 200).astype(np.float32)
cut_t = to_tensor(np.c_[xs_cut, np.full_like(xs_cut, ys)])
axes[1, 0].plot(xs_cut, target_fn(xs_cut, ys), color='k', lw=3, alpha=0.3, label='истина')
for name, model in reg_models.items():
    with torch.no_grad():
        axes[1, 0].plot(xs_cut, model(cut_t).cpu().numpy(), color=REG_COLORS[name], lw=1.5, label=name)
axes[1, 0].set_title(f'Срез при y = {ys}')
axes[1, 0].set_xlabel('x')
axes[1, 0].legend(fontsize=8)
for ax in list(axes[0]) + list(axes[1, 1:]):
    ax.set_aspect('equal')
    ax.grid(False)
plt.show()

In [ ]:
# --- 4.6 Функции на ребрах первого слоя KAN ---
kan = reg_models['KAN [2, 5, 1]']
x1d = torch.linspace(-1, 1, 200, device=device)
phi = kan.layers[0].edge_functions(x1d).cpu().numpy()
fig, axes = plt.subplots(2, phi.shape[2], figsize=(16, 5.2), sharex=True)
for i, var in enumerate(['x', 'y']):
    for j in range(phi.shape[2]):
        axes[i, j].plot(x1d.cpu().numpy(), phi[:, i, j], color='#eb6834')
        axes[i, j].set_title(f'phi({var} -> h{j + 1})', fontsize=10)
for ax in axes[1]:
    ax.set_xlabel('вход')
fig.suptitle('KAN, первый слой: функции на ребрах', y=1.0)
plt.tight_layout()
plt.show()

### Выводы по части 4

In [ ]:
r4 = results_part4
names4 = list(reg_models)
kan_name = 'KAN [2, 5, 1]'
best4 = r4['Test MSE'].idxmin()
border = (np.abs(GX) > 0.8) | (np.abs(GY) > 0.8)


def ep(v):
    return 'не достигнуто' if pd.isna(v) else f'{int(v)}'


text = ["- Функцию можно записать как сумму одномерных функций: sin(pi x) cos(pi y) = 0.5 sin(pi(x+y)) + "
        "0.5 sin(pi(x-y)), то есть f = 0.5 sin(pi(x+y)) + 0.5 sin(pi(x-y)) + x^2 - y^2. KAN как раз строит "
        "сумму одномерных функций, поэтому такая задача ему удобна.",
        "- Число параметров: " + ', '.join(f"{n} - {r4.loc[n, 'Параметров']}" for n in names4) + ".",
        "- Test MSE: " + ', '.join(f"{n} - {r4.loc[n, 'Test MSE']:.2e}" for n in names4)
        + f". Лучшая модель - {best4}."]
for other in names4:
    if other != kan_name:
        ratio = r4.loc[other, 'Test MSE'] / r4.loc[kan_name, 'Test MSE']
        if ratio >= 1:
            text.append(f"- KAN точнее, чем {other}, в {ratio:.1f} раза по MSE. Параметров у KAN "
                        f"{r4.loc[kan_name, 'Параметров']}, у {other} {r4.loc[other, 'Параметров']}.")
        else:
            text.append(f"- {other} точнее KAN в {1 / ratio:.1f} раза по MSE.")
for thr in ['1e-3', '1e-4']:
    col = f'Эпох до val MSE < {thr}'
    text.append(f"- Эпох до val MSE < {thr}: " + ', '.join(f"{n} - {ep(r4.loc[n, col])}" for n in names4) + ".")
t_ratio = r4.loc[kan_name, 'Время на эпоху, мс'] / r4.loc['MLP 3x16', 'Время на эпоху, мс']
text.append(f"- Эпоха KAN в {t_ratio:.1f} раза дороже эпохи MLP 3x16 ({r4.loc[kan_name, 'Время на эпоху, мс']:.1f} "
            f"против {r4.loc['MLP 3x16', 'Время на эпоху, мс']:.1f} мс), потому что для каждого входа считается базис "
            "B-сплайнов.")
s = "- Отношение средней ошибки у края квадрата (|x| или |y| > 0.8) к средней ошибке внутри: "
s += ', '.join(f"{n} - {np.abs(preds[n] - GF)[border].mean() / np.abs(preds[n] - GF)[~border].mean():.1f}"
               for n in names4) + "."
text.append(s)
text.append("- Функции на ребрах KAN гладкие, их можно нарисовать и посмотреть. Но при 5 скрытых нейронах "
            "разложение не единственное, поэтому в отдельных phi не обязательно видны чистые sin или x^2.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ИТОГОВЫЕ ВЫВОДЫ
# ============================================================
best1 = acc1.idxmax()
text = [
    f"1. На задаче \"круг vs кольцо\" лучшая модель - {best1} (test accuracy {acc1[best1]:.3f}). "
    f"Модель A с двумя ReLU-нейронами в слое получила {acc1['A']:.3f}, модель C с GELU того же размера - "
    f"{acc1['C']:.3f}."
    + (f" Для сети шириной 2 результат сильно зависит от seed: с ReLU от {width_acc['relu'][w2].min():.3f} "
       f"до {width_acc['relu'][w2].max():.3f}." if np.ptp(width_acc['relu'][w2]) > 0.05 else ""),
    f"2. Скрытые слои преобразуют плоскость так, что классы становятся линейно разделимыми: логистическая "
    f"регрессия на исходных координатах дает {raw_acc:.3f}, на эмбеддингах модели B - {probe_emb['B']:.3f}.",
    f"3. Шум снижает accuracy всех моделей (при sigma = 1.0 у B {noise_acc['B'][s10].mean():.3f}). "
    f"Изменение accuracy от 30% выбросов: B {-drops['B']:+.1f} п.п., A {-drops['A']:+.1f} п.п., C {-drops['C']:+.1f} п.п. "
    f"Сильнее всего на модель B влияет {top_phrase}.",
    f"4. На гладкой функции test MSE: " + ', '.join(f"{n} - {r4.loc[n, 'Test MSE']:.2e}" for n in names4)
    + f". Лучший результат у {best4}.",
    "5. Одной accuracy мало: модели с близкой accuracy могут иметь разную границу и разную устойчивость, "
    "а результаты маленьких сетей нужно проверять на нескольких seed.",
]
display(Markdown('\n'.join(text)))